# 🛡️ QuantumShield-FL: End-to-End Federated Demo

Demonstration of **Quantum-Secured Federated Learning** unifying:
1. **Quantum Key Distribution (BB84/E91)** for encrypting client parameter updates
2. **Differential Privacy (Gaussian/Laplace Mechanism)** for gradient-level protection
3. **Variational Quantum Classifier (VQC)** running on local client nodes
4. **Secure Aggregator (FedAvg / FedProx)** at the central server

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

from src.quantum.qkd import BB84Protocol, QKDSession
from src.quantum.qml import VQC
from src.federated import FederatedClient, FederatedServer, FedAvg

print('✅ QuantumShield-FL framework imported!')

## 1. Multi-Client Federated Setup

We simulate 4 hospital nodes participating in federated breast cancer diagnostic classification,
with each hospital encrypting its model updates via QKD keys.

In [ ]:
# Load data (2 primary features for 2-qubit VQC)
data = load_breast_cancer()
X, y = data.data[:, :2], data.target
X = StandardScaler().fit_transform(X) * (np.pi / 2)

n_clients = 4
splits = np.array_split(np.random.permutation(len(X)), n_clients)

clients = []
for i, idx in enumerate(splits):
    model = VQC(n_qubits=2, n_layers=2, learning_rate=0.05)
    client = FederatedClient(
        client_id=f'Hospital_{i+1}',
        model=model,
        X_train=X[idx],
        y_train=y[idx],
        local_epochs=3,
        use_qkd=True,
        use_dp=True,
        dp_epsilon=1.0
    )
    clients.append(client)
    print(f'🏥 {client.client_id} initialized with {len(idx)} private medical records.')

server = FederatedServer(clients=clients, aggregation='fedavg')
print('\n🏛️ Central Quantum Federated Server initialized.')

## 2. Federated Training Rounds with QKD Encryption

In [ ]:
n_rounds = 6
accuracies = []

print('🚀 Commencing Quantum-Secured Federated Training...')
for r in range(1, n_rounds + 1):
    metrics = server.run_round(round_num=r, verbose=False)
    acc = metrics['accuracy']
    accuracies.append(acc)
    print(f'  [Round {r}/{n_rounds}] Aggregated Global Accuracy: {acc*100:.2f}% | QKD Secured: ✅')

print(f'\n🎉 Training Complete! Final Global Accuracy: {accuracies[-1]*100:.2f}%')